# Experiment 12 — Multi-Dataset Generalization Validation

**Objective:** Prove that the FYDP-II Ensemble pipeline generalizes beyond the original cHL CODEX dataset.

| Dataset | Disease | Imaging | Cells | Markers | Cell Types |
|---------|---------|---------|-------|---------|------------|
| cHL CODEX (Original) | Hodgkin Lymphoma | CODEX | 145K | 49 | 16 |
| CRC CODEX | Colorectal Cancer | CODEX | 258K | 56 | ~25 |
| cHL MIBI | Hodgkin Lymphoma | MIBI | 1.67M | 41 | 14 |


In [1]:
import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning)

SEED = 7325111
BATCH_SIZE = 128

def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')


PyTorch: 2.10.0+cu128
Device: cuda


## MLP Infrastructure (From Experiment 12)


In [2]:
class CellDataset(Dataset):
    def __init__(self, X, y, is_train=True, mean=None, std=None):
        self.y = y
        if is_train:
            self.mean = np.mean(X, axis=0)
            self.std = np.std(X, axis=0) + 1e-8
        else:
            self.mean = mean
            self.std = std
        self.X = (X - self.mean) / self.std

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.float64),
                torch.tensor(self.y[idx], dtype=torch.long))

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes))

    def forward(self, x): return self.net(x)


## Reusable Pipeline Function


In [3]:
def run_experiment12_pipeline(X, y, feature_cols, class_names, dataset_name, max_cells=None):
    """Runs the full Experiment 12 pipeline on any dataset."""
    set_seed(SEED)
    NUM_FEATURES = X.shape[1]
    NUM_CLASSES = len(class_names)

    if max_cells and len(X) > max_cells:
        print(f"Subsampling {len(X)} -> {max_cells} cells (stratified)...")
        idx = np.arange(len(X))
        idx, _ = train_test_split(idx, train_size=max_cells, random_state=SEED, stratify=y)
        X, y = X[idx], y[idx]

    print(f'\n{"="*70}')
    print(f'RUNNING PIPELINE ON: {dataset_name}')
    print(f'{"="*70}')
    print(f'Cells: {len(X):,} | Features: {NUM_FEATURES} | Classes: {NUM_CLASSES}')

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y)

    # 1. MLP
    print("\n1. Training MLP...")
    ds_train = CellDataset(X_train, y_train, is_train=True)
    ds_valid = CellDataset(X_valid, y_valid, is_train=False, mean=ds_train.mean, std=ds_train.std)

    labels_list = y_train.tolist()
    n = float(len(labels_list))
    wpc = {c: n / labels_list.count(c) for c in sorted(set(labels_list))}
    sw = [wpc[l] for l in labels_list]
    sampler = WeightedRandomSampler(sw, len(sw))

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, num_workers=0)
    valid_loader = DataLoader(ds_valid, batch_size=BATCH_SIZE, sampler=SequentialSampler(ds_valid), drop_last=False, num_workers=0)

    model_mlp = MLP(NUM_FEATURES, 512, NUM_CLASSES, 0.10).to(DEVICE, dtype=torch.float64)
    optimizer = optim.Adam(model_mlp.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    best_loss, patience_counter, best_state = float('inf'), 0, None

    for epoch in range(500):
        model_mlp.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model_mlp(xb), yb)
            loss.backward()
            optimizer.step()

        model_mlp.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += loss_fn(model_mlp(xb), yb).item() * len(yb)
        val_loss /= len(ds_valid)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model_mlp.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f'   Early stop at epoch {epoch}')
                break

    model_mlp.load_state_dict(best_state)
    model_mlp.to(DEVICE).eval()
    mlp_probs_list = []
    with torch.no_grad():
        for xb, _ in valid_loader:
            logits = model_mlp(xb.to(DEVICE))
            mlp_probs_list.append(torch.softmax(logits, dim=1).cpu().numpy())
    mlp_probs = np.concatenate(mlp_probs_list)
    mlp_preds = np.argmax(mlp_probs, axis=1)
    mlp_f1 = f1_score(y_valid, mlp_preds, average='weighted')
    print(f'   MLP F1: {mlp_f1:.4f}')

    # 2. LightGBM
    print("2. Training LightGBM...")
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_valid_sc = scaler.transform(X_valid)
    lgb_train = lgb.Dataset(X_train_sc, label=y_train, feature_name=feature_cols)
    lgb_val = lgb.Dataset(X_valid_sc, label=y_valid, feature_name=feature_cols, reference=lgb_train)
    lgb_params = {
        'objective': 'multiclass', 'num_class': NUM_CLASSES,
        'metric': 'multi_logloss', 'num_leaves': 127, 'max_depth': 8,
        'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'is_unbalance': True, 'min_child_samples': 20,
        'lambda_l1': 0.1, 'lambda_l2': 0.1, 'verbose': -1, 'seed': SEED, 'n_jobs': -1}
    lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=1000,
        valid_sets=[lgb_val], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    lgb_probs = lgb_model.predict(X_valid_sc)
    lgb_preds = np.argmax(lgb_probs, axis=1)
    lgb_f1 = f1_score(y_valid, lgb_preds, average='weighted')
    print(f'   LightGBM F1: {lgb_f1:.4f}')

    # 3. XGBoost
    print("3. Training XGBoost...")
    sw_xgb = compute_sample_weight('balanced', y_train)
    xgb_params = {
        'objective': 'multi:softprob', 'num_class': NUM_CLASSES,
        'eval_metric': 'mlogloss', 'max_depth': 8, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5,
        'reg_alpha': 0.1, 'reg_lambda': 1.0, 'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed': SEED, 'verbosity': 0}
    dtrain = xgb.DMatrix(X_train_sc, label=y_train, weight=sw_xgb, feature_names=feature_cols)
    dvalid = xgb.DMatrix(X_valid_sc, label=y_valid, feature_names=feature_cols)
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=1000,
        evals=[(dvalid, 'valid')], early_stopping_rounds=50, verbose_eval=0)
    xgb_probs = xgb_model.predict(dvalid)
    xgb_preds = np.argmax(xgb_probs, axis=1)
    xgb_f1 = f1_score(y_valid, xgb_preds, average='weighted')
    print(f'   XGBoost F1: {xgb_f1:.4f}')

    # 4. Grid Search Ensemble
    print("4. Searching optimal ensemble weights...")
    best_gf1, best_w = 0, (1/3, 1/3, 1/3)
    for w1 in np.arange(0.1, 0.9, 0.1):
        for w2 in np.arange(0.1, 0.9 - w1, 0.1):
            w3 = 1.0 - w1 - w2
            if w3 < 0.05: continue
            cp = w1*mlp_probs + w2*lgb_probs + w3*xgb_probs
            cf1 = f1_score(y_valid, np.argmax(cp, axis=1), average='weighted')
            if cf1 > best_gf1: best_gf1, best_w = cf1, (w1, w2, w3)
    ens_probs = best_w[0]*mlp_probs + best_w[1]*lgb_probs + best_w[2]*xgb_probs
    ens_preds = np.argmax(ens_probs, axis=1)
    ens_f1 = f1_score(y_valid, ens_preds, average='weighted')

    # 5. UQ
    ps = np.stack([lgb_probs, xgb_probs, mlp_probs])
    uncertainty = ps.std(axis=0).max(axis=1)
    confidence = 1.0 - uncertainty
    uq_preds = ps.mean(axis=0).argmax(axis=1)
    uq_results = {}
    for t in [0.9, 0.8, 0.7, 0.6, 0.5]:
        mask = confidence >= t
        pct = mask.mean() * 100
        prec = (uq_preds[mask] == y_valid[mask]).mean() * 100 if mask.sum() > 0 else 0
        uq_results[t] = (pct, prec)

    # Print Results
    print(f'\n{"="*70}')
    print(f'RESULTS: {dataset_name}')
    print(f'{"="*70}')
    print(f'{"Model":<25} {"W-F1":>8}')
    print(f'{"-"*35}')
    print(f'  {"MLP":<23} {mlp_f1:8.4f}')
    print(f'  {"LightGBM":<23} {lgb_f1:8.4f}')
    print(f'  {"XGBoost":<23} {xgb_f1:8.4f}')
    print(f'  {"Grid Search Ensemble":<23} {ens_f1:8.4f}')
    print(f'  Weights: MLP={best_w[0]:.1f}, LGB={best_w[1]:.1f}, XGB={best_w[2]:.1f}')

    print(f'\n{"Confidence":<20} {"% Auto":>8} {"Precision":>10}')
    print(f'{"-"*40}')
    for t, (pct, prec) in uq_results.items():
        print(f'  >= {t:<16} {pct:7.1f}% {prec:9.1f}%')

    print(f'\nPer-Class Report (Ensemble):')
    print(classification_report(y_valid, ens_preds, target_names=class_names, digits=4))

    return {'dataset': dataset_name, 'mlp_f1': mlp_f1, 'lgb_f1': lgb_f1,
            'xgb_f1': xgb_f1, 'ens_f1': ens_f1, 'weights': best_w,
            'uq': uq_results, 'num_cells': len(X),
            'num_features': NUM_FEATURES, 'num_classes': NUM_CLASSES}


---
## Dataset 1: CRC CODEX (Colorectal Cancer)
Same imaging technology (CODEX), different disease → proves **cross-tissue generalization**.


In [4]:
print("Loading CRC CODEX dataset...")
df_crc = pd.read_csv("/kaggle/input/datasets/imranbhuiyan999/crc-clusters-neighborhoods-markers/CRC_clusters_neighborhoods_markers.csv")
print(f"Raw: {len(df_crc)} cells")

# Drop noisy / ambiguous classes
drop_classes = ['dirt', 'undefined', 'immune cells / vasculature', 'tumor cells / immune cells']
df_crc = df_crc[~df_crc['ClusterName'].isin(drop_classes)]
print(f"After cleanup: {len(df_crc)} cells, {df_crc['ClusterName'].nunique()} cell types")

# Identify marker columns (contain ':Cyc_')
meta_cols = ['Unnamed: 0','CellID','ClusterID','EventID','File Name','Region',
    'TMA_AB','TMA_12','Index in File','groups','patients','spots',
    'cell_id:cell_id','tile_nr:tile_nr','X:X','Y:Y',
    'X_withinTile:X_withinTile','Y_withinTile:Y_withinTile','Z:Z',
    'size:size','HOECHST1:Cyc_1_ch_1','DRAQ5:Cyc_23_ch_4',
    'Profile_Homogeneity:Fiter1','ClusterSize','ClusterName',
    'neighborhood10','neighborhood number final','neighborhood name']
binary_cols = [c for c in df_crc.columns if '+' in c]
exclude = set(meta_cols + binary_cols)
marker_cols_crc = [c for c in df_crc.columns if c not in exclude]
marker_names_crc = [c.split(' - ')[0].split(':')[0].strip() for c in marker_cols_crc]
print(f"Markers: {len(marker_cols_crc)}")

le_crc = LabelEncoder()
y_crc = le_crc.fit_transform(df_crc['ClusterName'].values)
class_names_crc = le_crc.classes_.tolist()
X_crc = df_crc[marker_cols_crc].values.astype(np.float32)
print(f"Ready: X={X_crc.shape}, Classes={len(class_names_crc)}")


Loading CRC CODEX dataset...
Raw: 258385 cells
After cleanup: 240554 cells, 25 cell types
Markers: 56
Ready: X=(240554, 56), Classes=25


### Run Pipeline on CRC CODEX


In [5]:
crc_results = run_experiment12_pipeline(
    X_crc, y_crc, marker_names_crc, class_names_crc,
    dataset_name="CRC CODEX (Colorectal Cancer)")



RUNNING PIPELINE ON: CRC CODEX (Colorectal Cancer)
Cells: 240,554 | Features: 56 | Classes: 25

1. Training MLP...
   Early stop at epoch 28
   MLP F1: 0.8636
2. Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[254]	valid_0's multi_logloss: 0.397729
   LightGBM F1: 0.8672
3. Training XGBoost...
   XGBoost F1: 0.8719
4. Searching optimal ensemble weights...

RESULTS: CRC CODEX (Colorectal Cancer)
Model                         W-F1
-----------------------------------
  MLP                       0.8636
  LightGBM                  0.8672
  XGBoost                   0.8719
  Grid Search Ensemble      0.8813
  Weights: MLP=0.4, LGB=0.4, XGB=0.2

Confidence             % Auto  Precision
----------------------------------------
  >= 0.9                 77.7%      95.3%
  >= 0.8                 89.8%      91.2%
  >= 0.7                 95.8%      89.3%
  >= 0.6                 98.8%      88.4%
  >= 0.5                100.0%  

---
## Dataset 2: cHL MIBI (Hodgkin Lymphoma — MIBI)
Same disease, different imaging technology → proves **cross-platform generalization**.


In [6]:
print("Loading cHL MIBI dataset...")
df_mibi = pd.read_csv("/kaggle/input/datasets/imranbhuiyan999/chl-1-mibi/cHL1_MIBI.csv")
print(f"Raw: {len(df_mibi)} cells")

mibi_meta = ['cellLabel','Annotation','centroidX','centroidY','cellSize','identifier']
marker_cols_mibi = [c for c in df_mibi.columns if c not in mibi_meta]
marker_names_mibi = marker_cols_mibi.copy()
print(f"Markers: {len(marker_cols_mibi)}, Cell types: {df_mibi['Annotation'].nunique()}")

le_mibi = LabelEncoder()
y_mibi = le_mibi.fit_transform(df_mibi['Annotation'].values)
class_names_mibi = le_mibi.classes_.tolist()
X_mibi = df_mibi[marker_cols_mibi].values.astype(np.float32)
print(f"Ready: X={X_mibi.shape}, Classes={len(class_names_mibi)}")


Loading cHL MIBI dataset...
Raw: 1669853 cells
Markers: 41, Cell types: 14
Ready: X=(1669853, 41), Classes=14


### Run Pipeline on cHL MIBI
> Subsampling to 200K cells for GPU memory.


In [7]:
mibi_results = run_experiment12_pipeline(
    X_mibi, y_mibi, marker_names_mibi, class_names_mibi,
    dataset_name="cHL MIBI (Hodgkin Lymphoma)", max_cells=200000)


Subsampling 1669853 -> 200000 cells (stratified)...

RUNNING PIPELINE ON: cHL MIBI (Hodgkin Lymphoma)
Cells: 200,000 | Features: 41 | Classes: 14

1. Training MLP...
   Early stop at epoch 26
   MLP F1: 0.8318
2. Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[334]	valid_0's multi_logloss: 0.214987
   LightGBM F1: 0.9206
3. Training XGBoost...
   XGBoost F1: 0.9188
4. Searching optimal ensemble weights...

RESULTS: cHL MIBI (Hodgkin Lymphoma)
Model                         W-F1
-----------------------------------
  MLP                       0.8318
  LightGBM                  0.9206
  XGBoost                   0.9188
  Grid Search Ensemble      0.9233
  Weights: MLP=0.1, LGB=0.5, XGB=0.4

Confidence             % Auto  Precision
----------------------------------------
  >= 0.9                 72.1%      96.8%
  >= 0.8                 85.3%      94.0%
  >= 0.7                 93.4%      92.5%
  >= 0.6                 9

---
## Cross-Dataset Comparison Table


In [8]:
orig = {'dataset':'cHL CODEX (Original)', 'mlp_f1':0.8643, 'lgb_f1':0.9022,
    'xgb_f1':0.8989, 'ens_f1':0.9056, 'num_cells':145161,
    'num_features':50, 'num_classes':16,
    'uq':{0.9:(77.9,96.9), 0.8:(85.4,95.0), 0.7:(90.8,93.2)}}

all_r = [orig, crc_results, mibi_results]

print('\n' + '='*90)
print('CROSS-DATASET GENERALIZATION RESULTS')
print('='*90)
hdr = f'{"Metric":<25}'
for r in all_r: hdr += f' | {r["dataset"][:22]:>22}'
print(hdr)
print('-'*90)

for label, key, fmt in [('Cells','num_cells','{:,.0f}'),('Features','num_features','{:d}'),
    ('Cell Types','num_classes','{:d}'),('MLP F1','mlp_f1','{:.4f}'),
    ('LightGBM F1','lgb_f1','{:.4f}'),('XGBoost F1','xgb_f1','{:.4f}'),
    ('Ensemble F1','ens_f1','{:.4f}')]:
    row = f'  {label:<23}'
    for r in all_r: row += f' | {fmt.format(r[key]):>22}'
    print(row)

print(f'\nUQ @ 0.8 Confidence:', end='')
for r in all_r:
    p,pr = r['uq'][0.8]
    print(f' | {p:.1f}% cov / {pr:.1f}% prec', end='')
print()

print(f'\n{"="*90}')
print('KEY FINDINGS:')
for r in all_r:
    gap = r['lgb_f1'] - r['mlp_f1']
    print(f'  {r["dataset"][:25]:<25}: LightGBM beats MLP by {gap*100:+.2f}pp')
tree_wins = all(r['lgb_f1'] > r['mlp_f1'] for r in all_r)
if tree_wins: print('\n  CONFIRMED: Trees consistently outperform MLPs across all datasets.')
print('='*90)



CROSS-DATASET GENERALIZATION RESULTS
Metric                    |   cHL CODEX (Original) | CRC CODEX (Colorectal  | cHL MIBI (Hodgkin Lymp
------------------------------------------------------------------------------------------
  Cells                   |                145,161 |                240,554 |                200,000
  Features                |                     50 |                     56 |                     41
  Cell Types              |                     16 |                     25 |                     14
  MLP F1                  |                 0.8643 |                 0.8636 |                 0.8318
  LightGBM F1             |                 0.9022 |                 0.8672 |                 0.9206
  XGBoost F1              |                 0.8989 |                 0.8719 |                 0.9188
  Ensemble F1             |                 0.9056 |                 0.8813 |                 0.9233

UQ @ 0.8 Confidence: | 85.4% cov / 95.0% prec | 89.8% cov / 91